<a href="https://colab.research.google.com/github/m4nn4t/stock_price_prediction/blob/MultiStockPricePrediction/MultiStock.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install requests pandas tqdm -q

In [2]:
import pandas as pd
import numpy as np
import requests

from tqdm import tqdm

In [3]:
API_KEY="76e79bea0310025bde0ea382ae51ecb7cc4d561f"

START="2018-01-01"

END="2025-06-03"

In [4]:
TICKERS=[

"AAPL",
"MSFT",
"GOOGL",
"AMZN",
"META",

"NVDA",
"TSLA",
"AMD",
"NFLX",
"INTC"

]

In [5]:
all_data=[]

for ticker in tqdm(TICKERS):

    try:

        url=f"https://api.tiingo.com/tiingo/daily/{ticker}/prices"

        params={
            "startDate":START,
            "endDate":END,
            "token":API_KEY
        }

        response=requests.get(
            url,
            params=params
        )

        data=response.json()

        df=pd.DataFrame(data)

        if len(df)==0:
            continue

        df["Ticker"]=ticker

        all_data.append(df)

    except Exception as e:

        print(
            ticker,
            e
        )

100%|██████████| 10/10 [00:05<00:00,  1.84it/s]


In [6]:
master_df=pd.concat(
    all_data,
    ignore_index=True
)

master_df["date"]=pd.to_datetime(
    master_df["date"]
)

master_df.head()

,date,close,high,low,open,volume,adjClose,adjHigh,adjLow,adjOpen,adjVolume,divCash,splitFactor,Ticker
0,2018-01-02 00:00:00+00:00,172.26,172.30,169.26,170.16,25048048,40.268882,40.278233,39.567578,39.777969,100192192,0.0,1.0,AAPL
1,2018-01-03 00:00:00+00:00,172.23,174.55,171.96,172.53,28819653,40.261869,40.804211,40.198751,40.331999,115278612,0.0,1.0,AAPL
2,2018-01-04 00:00:00+00:00,173.03,173.47,172.08,172.54,22211345,40.448883,40.551741,40.226804,40.334337,88845380,0.0,1.0,AAPL
3,2018-01-05 00:00:00+00:00,175.00,175.37,173.05,173.44,23016177,40.909406,40.995900,40.453559,40.544728,92064708,0.0,1.0,AAPL
4,2018-01-08 00:00:00+00:00,174.35,175.61,173.93,174.35,20134092,40.757457,41.052005,40.659275,40.757457,80536368,0.0,1.0,AAPL


In [7]:
print(master_df.shape)

print(
master_df["Ticker"].nunique()
)

(18650, 14)
10


In [8]:
master_df.to_csv(
    "multi_stock_raw.csv",
    index=False
)

print(
"Saved"
)

Saved


In [9]:
master_df.shape

master_df.head()

master_df["Ticker"].value_counts()

,count
Ticker,
AAPL,1865
MSFT,1865
GOOGL,1865
AMZN,1865
META,1865
NVDA,1865
TSLA,1865
AMD,1865
NFLX,1865


FEATURE ENGINEERING

In [10]:
master_df=master_df.sort_values(
    ["Ticker","date"]
)

In [11]:
def add_indicators(df):

    df=df.copy()

    # EMA

    df["EMA_10"]=df["close"].ewm(
        span=10,
        adjust=False
    ).mean()

    df["EMA_30"]=df["close"].ewm(
        span=30,
        adjust=False
    ).mean()

    # RSI

    delta=df["close"].diff()

    gain=delta.clip(lower=0)

    loss=-delta.clip(upper=0)

    avg_gain=gain.rolling(14).mean()

    avg_loss=loss.rolling(14).mean()

    rs=avg_gain/avg_loss

    df["RSI"]=100-(100/(1+rs))

    # MACD

    ema12=df["close"].ewm(
        span=12,
        adjust=False
    ).mean()

    ema26=df["close"].ewm(
        span=26,
        adjust=False
    ).mean()

    df["MACD"]=ema12-ema26

    # Bollinger Bands

    sma20=df["close"].rolling(20).mean()

    std20=df["close"].rolling(20).std()

    df["BB_UPPER"]=sma20+(2*std20)

    df["BB_LOWER"]=sma20-(2*std20)

    # ATR

    high_low=df["high"]-df["low"]

    high_close=np.abs(
        df["high"]-df["close"].shift()
    )

    low_close=np.abs(
        df["low"]-df["close"].shift()
    )

    ranges=pd.concat(
        [
            high_low,
            high_close,
            low_close
        ],
        axis=1
    )

    true_range=np.max(
        ranges,
        axis=1
    )

    df["ATR"]=true_range.rolling(14).mean()

    return df

In [12]:
feature_df=master_df.groupby(
    "Ticker",
    group_keys=False
).apply(
    add_indicators
)

/tmp/ipykernel_7775/2965470934.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ).apply(


In [13]:
feature_df=feature_df.dropna()

feature_df.shape

(18460, 21)

In [14]:
feature_df.columns

Index(['date', 'close', 'high', 'low', 'open', 'volume', 'adjClose', 'adjHigh',
       'adjLow', 'adjOpen', 'adjVolume', 'divCash', 'splitFactor', 'Ticker',
       'EMA_10', 'EMA_30', 'RSI', 'MACD', 'BB_UPPER', 'BB_LOWER', 'ATR'],
      dtype='object')

In [15]:
feature_df.to_csv(
    "multi_stock_features.csv",
    index=False
)

print("Feature dataset saved")

Feature dataset saved


In [16]:
feature_df.shape

feature_df.head()

feature_df.columns

Index(['date', 'close', 'high', 'low', 'open', 'volume', 'adjClose', 'adjHigh',
       'adjLow', 'adjOpen', 'adjVolume', 'divCash', 'splitFactor', 'Ticker',
       'EMA_10', 'EMA_30', 'RSI', 'MACD', 'BB_UPPER', 'BB_LOWER', 'ATR'],
      dtype='object')

UNIVERSAL TRAINING DATASET

To build a scalable stock prediction platform, a universal dataset was created using historical data from ten major stocks: AAPL, MSFT, GOOGL, AMZN, META, NVDA, TSLA, AMD, NFLX, and INTC. Instead of relying solely on raw market data (Open, High, Low, Close, and Volume), several technical indicators such as EMA-10, EMA-30, RSI, MACD, Bollinger Bands, and ATR were added to capture trend, momentum, and volatility information. Each stock was assigned a unique Stock_ID, allowing the model to distinguish between different company behaviors. The dataset was then split chronologically to prevent future data leakage, ensuring a realistic forecasting setup. Using a sliding window approach, the previous 100 days of market and indicator data were used as input features to predict the next 3 days of closing prices. This process generated 13,730 training samples and 2,670 testing samples, creating a robust multi-stock dataset that enables the model to learn general market patterns rather than the behavior of a single stock, making it suitable for deployment in a real-world stock forecasting platform.

In [17]:
#creating stock ID for individual companies
from sklearn.preprocessing import LabelEncoder

encoder=LabelEncoder()

feature_df["Stock_ID"]=encoder.fit_transform(
    feature_df["Ticker"]
)

feature_df[
    ["Ticker","Stock_ID"]
].drop_duplicates().sort_values(
    "Stock_ID"
)

,Ticker,Stock_ID
19,AAPL,0
13074,AMD,1
5614,AMZN,2
3749,GOOGL,3
16804,INTC,4
7479,META,5
1884,MSFT,6
14939,NFLX,7
9344,NVDA,8
11209,TSLA,9


In [18]:
FEATURE_COLUMNS=[

"open",
"high",
"low",
"close",
"volume",

"EMA_10",
"EMA_30",

"RSI",

"MACD",

"BB_UPPER",
"BB_LOWER",

"ATR",

"Stock_ID"

]

In [19]:
#normalize Features
from sklearn.preprocessing import MinMaxScaler

scaler=MinMaxScaler()

feature_df[FEATURE_COLUMNS]=scaler.fit_transform(
    feature_df[FEATURE_COLUMNS]
)

In [20]:
WINDOW=100

FORECAST=3

In [21]:
X_train=[]

y_train=[]

X_test=[]

y_test=[]

In [22]:
for ticker in feature_df["Ticker"].unique():

    stock_df=feature_df[
        feature_df["Ticker"]==ticker
    ].sort_values("date")

In [23]:
    split_idx=int(
        len(stock_df)*0.8
    )

    train_df=stock_df.iloc[:split_idx]

    test_df=stock_df.iloc[split_idx:]

In [24]:
    train_values=train_df[
        FEATURE_COLUMNS
    ].values

    close_idx=FEATURE_COLUMNS.index(
        "close"
    )

In [27]:
    for i in range(
        len(train_values)-WINDOW-FORECAST
    ):

        X_train.append(
            train_values[
                i:i+WINDOW
            ]
        )

        y_train.append(
            train_values[
                i+WINDOW:
                i+WINDOW+FORECAST,
                close_idx
            ]
        )

In [28]:
    test_values=test_df[
        FEATURE_COLUMNS
    ].values

In [29]:
    for i in range(
        len(test_values)-WINDOW-FORECAST
    ):

        X_test.append(
            test_values[
                i:i+WINDOW
            ]
        )

        y_test.append(
            test_values[
                i+WINDOW:
                i+WINDOW+FORECAST,
                close_idx
            ]
        )

In [30]:
X_train=np.array(X_train)

y_train=np.array(y_train)

X_test=np.array(X_test)

y_test=np.array(y_test)

In [31]:
print("X_train:",X_train.shape)

print("y_train:",y_train.shape)

print("X_test:",X_test.shape)

print("y_test:",y_test.shape)

X_train: (1373, 100, 13)
y_train: (1373, 3)
X_test: (267, 100, 13)
y_test: (267, 3)


In [32]:
print(feature_df.shape)

print(feature_df["Ticker"].value_counts())

(18460, 22)
Ticker
AAPL     1846
AMD      1846
AMZN     1846
GOOGL    1846
INTC     1846
META     1846
MSFT     1846
NFLX     1846
NVDA     1846
TSLA     1846
Name: count, dtype: int64


In [33]:
for ticker in feature_df["Ticker"].unique():

    stock_df=feature_df[
        feature_df["Ticker"]==ticker
    ]

    print(
        ticker,
        len(stock_df)
    )

AAPL 1846
AMD 1846
AMZN 1846
GOOGL 1846
INTC 1846
META 1846
MSFT 1846
NFLX 1846
NVDA 1846
TSLA 1846


In [34]:
print(
feature_df["Ticker"].nunique()
)

10


In [35]:
print(feature_df.shape)

print(feature_df["Ticker"].nunique())

print(feature_df["Ticker"].value_counts())

(18460, 22)
10
Ticker
AAPL     1846
AMD      1846
AMZN     1846
GOOGL    1846
INTC     1846
META     1846
MSFT     1846
NFLX     1846
NVDA     1846
TSLA     1846
Name: count, dtype: int64


In [36]:
X_train=[]
y_train=[]

X_test=[]
y_test=[]

In [37]:
WINDOW=100
FORECAST=3

X_train=[]
y_train=[]

X_test=[]
y_test=[]

for ticker in feature_df["Ticker"].unique():

    stock_df=feature_df[
        feature_df["Ticker"]==ticker
    ].sort_values("date")

    split_idx=int(len(stock_df)*0.8)

    train_df=stock_df.iloc[:split_idx]

    test_df=stock_df.iloc[split_idx:]

    train_values=train_df[FEATURE_COLUMNS].values

    test_values=test_df[FEATURE_COLUMNS].values

    close_idx=FEATURE_COLUMNS.index("close")

    for i in range(len(train_values)-WINDOW-FORECAST):

        X_train.append(
            train_values[i:i+WINDOW]
        )

        y_train.append(
            train_values[
                i+WINDOW:i+WINDOW+FORECAST,
                close_idx
            ]
        )

    for i in range(len(test_values)-WINDOW-FORECAST):

        X_test.append(
            test_values[i:i+WINDOW]
        )

        y_test.append(
            test_values[
                i+WINDOW:i+WINDOW+FORECAST,
                close_idx
            ]
        )


X_train=np.array(X_train)

y_train=np.array(y_train)

X_test=np.array(X_test)

y_test=np.array(y_test)

print(X_train.shape)
print(y_train.shape)

print(X_test.shape)
print(y_test.shape)

(13730, 100, 13)
(13730, 3)
(2670, 100, 13)
(2670, 3)
